In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import zscore
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.cluster import KMeans
from xgboost import XGBRegressor
import matplotlib.pyplot as plt
import plotly.express as px
import dash
import dash_core_components as dcc
import dash_html_components as html
from dash.dependencies import Input, Output

# armamento del Data Frame proveniente de scikit-learn

data = datasets.fetch_california_housing()

df = pd.DataFrame(data["data"],columns=data["feature_names"])
df["MedHouseVal"] = data["target"]
df["MedHouseVal"] = df["MedHouseVal"] * 100000

df_model = df.copy() # generando una copia, asi no afectamos al Data Frame original

# removiendo los valores atípicos para la optimización del modelo

def remove_outliers(df,columns):
    for c in columns:
        df[c] = df[c].mask(zscore(df[c]).abs() > 3, np.nan)
    
    return df

df_model = remove_outliers(df_model,df.columns)

df_model.dropna(inplace=True)

df_model 

#### Boosting: anteriormente habiamos mencionado la técnica de Bagging, donde generábamos distintos modelos entrenados de forma independiente, en este caso cada modelo optmizará al siguiente estimando cada vez más la precisión final.
Random Search CV: función en la que se crean modelos con los diferentes parámetros otorgados para luego elegir aleatoriamente algunos de ellos, realizar validaciones cruzadas(X cantidad de pruebas con distintos datos de entrenamiento) y obtener un promedio del rendimiento(R2_score) de cada uno

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(df_model[data["feature_names"]],
                                                    df_model["MedHouseVal"],
                                                    test_size=0.25)

# ajuste de hiperparámetros utilizando Random Search 

xgbr_test = XGBRegressor()           

turned_parameters = {
    "n_estimators":[100,200,300],
    "max_depth":[3,4,5],
    "learning_rate":[0.3,0.4,0.5],
    "min_child_weight":[1,2,3]
}

grid_search = RandomizedSearchCV(xgbr_test,turned_parameters,cv=5)
grid_search.fit(x_train,y_train)

# asignandole al modelo los parámetros que obtuvieron mejores resultados

xgbr = XGBRegressor(n_estimators = grid_search.best_params_["n_estimators"],
                    max_depth = grid_search.best_params_["max_depth"],
                    learning_rate = grid_search.best_params_["learning_rate"],
                    min_child_weight = grid_search.best_params_["min_child_weight"])    


model = xgbr.fit(x_train,y_train)

# validando la eficiencia del modelo con los datos de prueba

print(f"MAE: {mean_absolute_error(x_test, y_test)}")
print(f"MSE: {mean_squared_error(x_test, y_test)}")
print(f"RMSE: {mean_squared_error(x_test, y_test, squared=True)}")

### Métricas de rendimiento para modelos de regresión
Tienen mucha similitud con las medidas de variabilidad en cuanto a el método matemático que emplean, utilizan los residuos del modelo, estos son las diferencias entre las predicciones del modelo regresor y los valores del modelo real. Aglunos ejemplos de estas métricas:

Error absoluto medio: similar a la desviacíon media absoluta, es la división por el número de muestra de la sumatoria de los residuos absolutos.

Error cuadrático medio: similar a la varianza, es la división por el número de muestra de la sumatoria de los residuos al cuadrado.

Raíz cuadrada del error cuadrático medio: similar a la desviación éstandar, se utiliza para convertir a la unidad en la que se basan los valores.

Coeficiente de determinación(R2 Score): valor de 0 a 1 que evalúa la bondad de ajuste de nuestro modelo a los datos, pero solo utilizada en modelos lineales.

In [ ]:
app = dash.Dash(__name__)

app.layout = html.Div(id="body",className="e5_body",children=[
    html.H1("Viviendas en California ",id="title",className="e5_title"),
        html.Div(id="div",className="e5_div",children=[
            dcc.Dropdown(id="dropdown",className="e5_dropdown",
                        options = [
                            {"label":"Ingreso medio","value":"MedInc"},
                            {"label":"Edad media","value":"HouseAge"},
                            {"label":"Promedio de habitaciones","value":"AveRooms"},
                            {"label":"Promedio de dormitorios","value":"AveBedrms"},
                            {"label":"Población","value":"Population"},
                            {"label":"Promedio de ocupación","value":"AveOccuption"}
                        ],
                        value="MedInc",
                        multi=False,
                        clearable=False)]),
            dcc.Graph(id="graph",className="e5_graph",figure={})
])

@app.callback(
    Output(component_id="graph",component_property="figure"),
    [Input(component_id="dropdown",component_property="value")]
)

def update_graph(slct_var):
    
    scatter_map = px.scatter(df,x="Longitude",y="Latitude",color=slct_var)
    
    return scatter_map
    
if __name__ == "__main__":
    app.run_server(debug=False)

In [ ]:
clusters = []
inertias = []

for c in range(3,12):
    kmeans = KMeans(n_clusters=c).fit(df["MedHouseVal"].values.reshape((-1,1)))
    clusters.append(c)
    inertias.append(kmeans.inertia_)
    
plt.plot(clusters,inertias,marker="o")
plt.grid("on")
plt.show()

### Método del codo
método que se realiza a la hora de definir el número de centroides de un modelo de clusterización, dónde el objetivo es generar la menor cantidad de centroides y obtener la menor inercia(alejamiento entre los miembros de clusters y su centroide)
__________________________________________________________________________________________________________________________________________________________________________

#### Aprendizaje no supervisado
las observaciones no tienen una respuesta asociada que guíe el aprendizaje, uno de sus algoritmos es Kmeans y genera sus propias etiquetas determinando los grupos de obejtos más parecidos según sus características

In [ ]:
kmeans = KMeans(n_clusters=5).fit(df["MedHouseVal"].values.reshape((-1,1)))

clusters = kmeans.labels_

df["clusters"] = clusters

range_values = np.array([])

for c in df["clusters"].sort_values().unique():
    cluster = df.loc[df["clusters"] == c,["clusters","MedHouseVal"]]
    max_value = str(cluster["MedHouseVal"].max())
    min_value = str(cluster["MedHouseVal"].min())
    range_values = np.append(range_values,min_value)
    range_values = np.append(range_values,max_value)
    
range_values = range_values.reshape((-1,2))
    
df["clusters"] = df["clusters"].replace(
    {
        0:f"0 ({range_values[0,0][:8]}$-{range_values[0,1][:8]}$)",
        1:f"1 ({range_values[1,0][:8]}$-{range_values[1,1][:8]}$)",
        2:f"2 ({range_values[2,0][:8]}$-{range_values[2,1][:8]}$)",
        3:f"3 ({range_values[3,0][:8]}$-{range_values[3,1][:8]}$)",
        4:f"4 ({range_values[4,0][:8]}$-{range_values[4,1][:8]}$)"
    })


fig = px.scatter(df,x="Longitude",y="Latitude",color="clusters")
fig.update_layout(title="Range of Houses'values")
fig 

##### Grupos de precios de las viviendas
Ya tenemos una idea clara en cuanto a los rangos de precios que las viviendas suelen frecuentar y como su ubicación influye, por ejemplo: vemos un evindente aumento en las viviendas más cercanas a la costa de California, esto ubicando los cluster donde dentro de estos se encuentran los valores más caros siendo (296.800-414.100) y (414.300-500.000), también, observamos que los precios suelen valer entre 15.000 y 127.800 gracias al cluster que guarda esos valores y siendo el más numeroso